# Chapter 1 — How LLMs Process Customer Journey Signals

**Stage:** Foundations | **Book:** *Mastering Agentic AI for Customer Journey Marketing*

---

## What You Will Learn

| Concept | Why It Matters |
|---|---|
| Journey signals | Page views, email opens, form fills are the raw fuel for intent classification |
| LLM-based intent scoring | Replace brittle rule-based scoring with contextual understanding |
| Multi-provider patterns | OpenAI, Anthropic (Claude), and Google (Gemini) each process the same signal data |
| Mock-first development | Build and test pipelines before connecting live APIs |

This notebook demonstrates how to feed raw customer journey events into Large Language Models
and receive structured intent classifications that map prospects to the six journey stages:
**Awareness, Consideration, Decision, Onboarding, Retention, and Advocacy**.

In [ ]:
# File      : 01_journey_signals_and_llms.ipynb
# Stage     : Foundations
# Chapter   : Chapter 1 - Journey Signals and LLMs
# Framework : OpenAI, Anthropic, Google Generative AI
# Author    : Pushparajan Ramar
# Repo      : https://github.com/Pushparajan/agenticai-marketing

import os
import json
from datetime import datetime, timedelta
from typing import Any

USE_MOCK = os.getenv("USE_MOCK_APIS", "true").lower() == "true"
print(f"USE_MOCK = {USE_MOCK}")

## 1 — Mock Journey Events for a B2B SaaS Prospect

We simulate a prospect ("Acme Corp") moving through stages over 14 days.
Each event carries a `type`, `timestamp`, and `metadata` dict.

In [ ]:
BASE_DATE = datetime(2026, 3, 1)

JOURNEY_EVENTS = [
    # --- Awareness signals ---
    {
        "event_id": "evt_001",
        "contact_email": "jdoe@acmecorp.com",
        "type": "page_view",
        "timestamp": (BASE_DATE + timedelta(days=0, hours=9)).isoformat(),
        "metadata": {"url": "/blog/ai-customer-journey", "duration_sec": 185}
    },
    {
        "event_id": "evt_002",
        "contact_email": "jdoe@acmecorp.com",
        "type": "page_view",
        "timestamp": (BASE_DATE + timedelta(days=0, hours=9, minutes=5)).isoformat(),
        "metadata": {"url": "/resources/martech-guide", "duration_sec": 240}
    },
    # --- Consideration signals ---
    {
        "event_id": "evt_003",
        "contact_email": "jdoe@acmecorp.com",
        "type": "email_open",
        "timestamp": (BASE_DATE + timedelta(days=2, hours=10)).isoformat(),
        "metadata": {"campaign": "nurture_series_1", "subject": "See how AI boosts retention"}
    },
    {
        "event_id": "evt_004",
        "contact_email": "jdoe@acmecorp.com",
        "type": "page_view",
        "timestamp": (BASE_DATE + timedelta(days=3, hours=14)).isoformat(),
        "metadata": {"url": "/pricing", "duration_sec": 120}
    },
    {
        "event_id": "evt_005",
        "contact_email": "jdoe@acmecorp.com",
        "type": "page_view",
        "timestamp": (BASE_DATE + timedelta(days=3, hours=14, minutes=3)).isoformat(),
        "metadata": {"url": "/case-studies/fintech-retention", "duration_sec": 310}
    },
    # --- Decision signals ---
    {
        "event_id": "evt_006",
        "contact_email": "jdoe@acmecorp.com",
        "type": "form_fill",
        "timestamp": (BASE_DATE + timedelta(days=5, hours=11)).isoformat(),
        "metadata": {"form": "demo_request", "company_size": "200-500", "role": "VP Marketing"}
    },
    {
        "event_id": "evt_007",
        "contact_email": "jdoe@acmecorp.com",
        "type": "email_click",
        "timestamp": (BASE_DATE + timedelta(days=7, hours=9)).isoformat(),
        "metadata": {"campaign": "demo_followup", "link": "/book-meeting"}
    },
    # --- Post-decision signals ---
    {
        "event_id": "evt_008",
        "contact_email": "jdoe@acmecorp.com",
        "type": "page_view",
        "timestamp": (BASE_DATE + timedelta(days=10, hours=16)).isoformat(),
        "metadata": {"url": "/docs/getting-started", "duration_sec": 450}
    },
    {
        "event_id": "evt_009",
        "contact_email": "jdoe@acmecorp.com",
        "type": "feature_use",
        "timestamp": (BASE_DATE + timedelta(days=12, hours=10)).isoformat(),
        "metadata": {"feature": "journey_builder", "actions_taken": 14}
    },
    {
        "event_id": "evt_010",
        "contact_email": "jdoe@acmecorp.com",
        "type": "nps_response",
        "timestamp": (BASE_DATE + timedelta(days=14, hours=15)).isoformat(),
        "metadata": {"score": 9, "comment": "Love the AI recommendations!"}
    },
]

print(f"Total journey events: {len(JOURNEY_EVENTS)}")
for evt in JOURNEY_EVENTS:
    print(f"  {evt['timestamp'][:10]}  {evt['type']:15s}  {evt['metadata']}")

## 2 — Intent Classification Prompt

We use the same system prompt and user payload for all three providers.
The LLM returns a structured JSON response with:
- `current_stage` — one of the six journey stages
- `confidence` — 0.0 to 1.0
- `signals_used` — which events drove the classification
- `next_best_action` — recommended marketing action

In [ ]:
SYSTEM_PROMPT = """You are a B2B SaaS marketing analyst AI. Given a chronological list of
customer journey events, classify the contact's current journey stage and recommend a
next-best action.

Journey stages (in order):
1. Awareness   — first touch, content consumption, brand discovery
2. Consideration — comparing options, viewing pricing, reading case studies
3. Decision     — demo requests, meeting bookings, proposal engagement
4. Onboarding   — docs visits, initial feature usage, setup activities
5. Retention    — ongoing feature usage, support interactions, health scores
6. Advocacy     — NPS promoters, referrals, review submissions

Return ONLY valid JSON with these fields:
{
  "current_stage": "<stage_name>",
  "confidence": <float 0-1>,
  "signals_used": ["<event_id>", ...],
  "reasoning": "<one sentence>",
  "next_best_action": "<recommended action>"
}"""


def build_user_message(events: list[dict]) -> str:
    """Format journey events as a user message for the LLM."""
    header = f"Contact: {events[0]['contact_email']}\nJourney events ({len(events)} total):\n\n"
    body = json.dumps(events, indent=2)
    return header + body


USER_MESSAGE = build_user_message(JOURNEY_EVENTS)
print(USER_MESSAGE[:500], "\n...")

## 3 — Mock LLM Responses

When `USE_MOCK=True`, we return pre-built responses so the notebook runs
without API keys. The structure mirrors what each provider returns.

In [ ]:
MOCK_CLASSIFICATION = {
    "current_stage": "Advocacy",
    "confidence": 0.88,
    "signals_used": ["evt_009", "evt_010"],
    "reasoning": (
        "The contact has completed onboarding (docs + feature usage) and submitted "
        "an NPS score of 9, indicating strong promoter potential."
    ),
    "next_best_action": (
        "Send a personalized referral-program invitation and request a G2 review."
    ),
}

print(json.dumps(MOCK_CLASSIFICATION, indent=2))

## 4 — OpenAI (GPT-4.1) Intent Classification

In [ ]:
def classify_with_openai(events: list[dict]) -> dict:
    """Classify journey stage using OpenAI GPT-4.1."""
    if USE_MOCK:
        print("[mock] OpenAI GPT-4.1 — returning cached response")
        return MOCK_CLASSIFICATION

    from openai import OpenAI

    client = OpenAI()  # reads OPENAI_API_KEY from env
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_message(events)},
        ],
        temperature=0.2,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)


openai_result = classify_with_openai(JOURNEY_EVENTS)
print("\n=== OpenAI GPT-4.1 Classification ===")
print(json.dumps(openai_result, indent=2))

## 5 — Anthropic (Claude) Intent Classification

In [ ]:
def classify_with_claude(events: list[dict]) -> dict:
    """Classify journey stage using Anthropic Claude."""
    if USE_MOCK:
        print("[mock] Anthropic Claude — returning cached response")
        return MOCK_CLASSIFICATION

    import anthropic

    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=[
            {"role": "user", "content": build_user_message(events)},
        ],
        temperature=0.2,
    )
    return json.loads(response.content[0].text)


claude_result = classify_with_claude(JOURNEY_EVENTS)
print("\n=== Anthropic Claude Classification ===")
print(json.dumps(claude_result, indent=2))

## 6 — Google (Gemini) Intent Classification

In [ ]:
def classify_with_gemini(events: list[dict]) -> dict:
    """Classify journey stage using Google Gemini."""
    if USE_MOCK:
        print("[mock] Google Gemini — returning cached response")
        return MOCK_CLASSIFICATION

    import google.generativeai as genai

    genai.configure()  # reads GOOGLE_API_KEY from env
    model = genai.GenerativeModel(
        model_name="gemini-2.5-pro",
        system_instruction=SYSTEM_PROMPT,
        generation_config={"temperature": 0.2, "response_mime_type": "application/json"},
    )
    response = model.generate_content(build_user_message(events))
    return json.loads(response.text)


gemini_result = classify_with_gemini(JOURNEY_EVENTS)
print("\n=== Google Gemini Classification ===")
print(json.dumps(gemini_result, indent=2))

## 7 — Side-by-Side Comparison

In [ ]:
import pandas as pd

results = {
    "OpenAI GPT-4.1": openai_result,
    "Anthropic Claude": claude_result,
    "Google Gemini": gemini_result,
}

comparison_rows = []
for provider, result in results.items():
    comparison_rows.append({
        "Provider": provider,
        "Stage": result.get("current_stage", "N/A"),
        "Confidence": result.get("confidence", 0.0),
        "Signals Used": ", ".join(result.get("signals_used", [])),
        "Next Best Action": result.get("next_best_action", "N/A"),
    })

df_comparison = pd.DataFrame(comparison_rows)
print("\n=== Provider Comparison ===")
print(df_comparison.to_string(index=False))

## 8 — Processing Sub-Sequences (Stage Progression)

In [ ]:
# Classify the prospect at different points in time to show stage progression
stage_snapshots = [
    ("Day 1 (2 events)",  JOURNEY_EVENTS[:2]),
    ("Day 3 (5 events)",  JOURNEY_EVENTS[:5]),
    ("Day 7 (7 events)",  JOURNEY_EVENTS[:7]),
    ("Day 12 (9 events)", JOURNEY_EVENTS[:9]),
    ("Day 14 (all 10)",   JOURNEY_EVENTS[:10]),
]

MOCK_PROGRESSION = [
    {"current_stage": "Awareness",     "confidence": 0.92},
    {"current_stage": "Consideration", "confidence": 0.85},
    {"current_stage": "Decision",      "confidence": 0.90},
    {"current_stage": "Retention",     "confidence": 0.78},
    {"current_stage": "Advocacy",      "confidence": 0.88},
]

print("=== Stage Progression Over Time ===")
print(f"{'Snapshot':<22} {'Stage':<16} {'Confidence'}")
print("-" * 52)
for i, (label, events_slice) in enumerate(stage_snapshots):
    if USE_MOCK:
        result = MOCK_PROGRESSION[i]
    else:
        result = classify_with_openai(events_slice)
    print(f"{label:<22} {result['current_stage']:<16} {result['confidence']:.2f}")

---

## Key Takeaways

1. **Journey signals are structured events** — page views, email opens, form fills, and product
   usage data each carry intent signals that LLMs can interpret contextually.

2. **LLMs replace brittle scoring rules** — instead of hand-coded point thresholds, a single
   prompt produces stage classification, reasoning, and next-best-action recommendations.

3. **Multi-provider consistency** — OpenAI, Claude, and Gemini all handle the same payload;
   the wrapper pattern makes it easy to swap or ensemble providers.

4. **Stage progression tracking** — by feeding growing event sequences, you can watch a
   prospect move from Awareness through Advocacy over time.

**Next:** [02_async_batch_processing.ipynb](./02_async_batch_processing.ipynb) — scale this
pattern to 500 contacts using async batch processing.